# Lesson 10.1: What Is Agentic RAG and How Is It Different from Standard RAG?

**Companion notebook for Lesson 10.1 — Module 10: Agentic RAG**

---

| Section | What you will build |
|---|---|
| 1. Standard RAG Fails on Compound Questions | Prove it with cosine similarity numbers |
| 2. Embedding Collapse on Multi-Intent Queries | Visualise how compound queries go blurry |
| 3. Component 1 — The Planner | Question decomposition: one LLM call, JSON output |
| 4. Component 2 — The Retriever | Query formulation + keyword search (no DB needed) |
| 5. Component 3 — The Critic | Sufficiency scoring: does this context answer the sub-question? |
| 6. Component 4 — The Synthesiser | Labelled context in, coherent comparison out |
| 7. The Agent Loop | All four components wired together with a `max_retries` ceiling |
| 8. Execution Trace Visualisation | See the loop run; count the LLM calls |
| 9. When to Use Agentic RAG | Decision helper: route by question complexity |
| 10. Cost Comparison | LLM call count + token math; standard vs agentic |
| 11. Routing Classifier | Simple classifier that picks standard vs agentic path |
| 12. Claude API Version | Drop-in replacement using `anthropic` |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`  
**Optional (Sections 3–7 real mode, 12):** `anthropic`

> Sections 3–7 run in **mock mode** by default (no API key needed).
> Set `ANTHROPIC_API_KEY` and `USE_REAL_LLM = True` to run with a live LLM.


In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib
# !pip install anthropic   # optional — Sections 3-7 real mode and Section 12

%matplotlib inline
import os, json, time, textwrap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS']        = '1'

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3

# ── Toggle: True = live Claude API calls; False = deterministic mock ──────
USE_REAL_LLM = False

ANTHROPIC_AVAILABLE = False
try:
    import anthropic
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass

if USE_REAL_LLM and not ANTHROPIC_AVAILABLE:
    print('WARNING: USE_REAL_LLM=True but anthropic not available.')
    print('Falling back to mock mode. Set ANTHROPIC_API_KEY to enable real calls.')
    USE_REAL_LLM = False

mode = 'REAL (Claude API)' if USE_REAL_LLM else 'MOCK (deterministic)'
print(f'LLM mode: {mode}')
print('Imports ready.')


In [ ]:
# ── Shared knowledge base (used by all sections) ─────────────────────────────
# Mirrors the blog's fake vector DB exactly.
# In production: replace fake_vector_search with a call to Pinecone, pgvector, etc.

KNOWLEDGE_BASE = {
    'our_refund': (
        'Our refund policy: refunds within 14 days of purchase. '
        'Digital goods are non-refundable. No restocking fee.'
    ),
    'compA_refund': (
        'Competitor A refund policy: 30-day refund window. '
        '10% restocking fee on opened items.'
    ),
    'compB_refund': (
        'Competitor B refund policy: 7-day refund window. '
        'Refunds only for defective items.'
    ),
    'shipping': (
        'Shipping policy: free shipping over $50. '
        'Delivery in 3-5 days.'
    ),
    'privacy': (
        'Privacy policy: we do not sell user data. '
        'Data is retained for 24 months unless deletion is requested.'
    ),
    'pricing': (
        'Pricing: Free tier includes core features. '
        'Pro plan is $29/month. Enterprise pricing is custom.'
    ),
}


def fake_vector_search(query: str, top_k: int = 2) -> list:
    """Keyword overlap search standing in for a real vector DB."""
    query_words = set(query.lower().split())
    scored = []
    for doc_id, text in KNOWLEDGE_BASE.items():
        score = len(query_words & set(text.lower().split()))
        scored.append((score, doc_id, text))
    scored.sort(reverse=True)
    return [(doc_id, text) for _, doc_id, text in scored[:top_k]]


print(f'Knowledge base: {len(KNOWLEDGE_BASE)} documents')
for doc_id, text in KNOWLEDGE_BASE.items():
    print(f'  [{doc_id}] {text[:70]}...')


---
## 1. Standard RAG Fails on Compound Questions

Standard RAG does this:

```
1. Embed the whole question as one vector
2. Find top-k similar chunks
3. LLM answers from those chunks
```

For a compound question like:
> *"Compare our refund policy with Competitor A and Competitor B. Where are we stricter?"*

...that single embedding has to represent **four** things simultaneously:
our policy, Competitor A, Competitor B, and the comparison dimension.
It ends up representing none of them well.

Let's prove this with retrieval scores.

**The developer is in control in standard RAG. The LLM only generates at the end.**

```python
# Standard RAG (developer decides everything)
chunks = vector_db.search(user_query)       # YOU decide
answer = llm.generate(user_query, chunks)   # YOU decide what to pass
```

```python
# Agentic RAG (LLM decides what to retrieve and when)
tools  = [search_vector_db, search_web, lookup_database]
answer = llm.run(user_query, tools=tools)   # LLM decides
```


In [ ]:
# ── Standard RAG: one query, one retrieval ───────────────────────────────────

COMPOUND_QUERY = (
    'Compare our refund policy with Competitor A and Competitor B. '
    'Where are we stricter?'
)

FOCUSED_QUERIES = [
    'What is our refund policy?',
    'What is Competitor A refund policy?',
    'What is Competitor B refund policy?',
]

print('=== Standard RAG: one compound query ===\n')
print(f'Query: "{COMPOUND_QUERY}"\n')

compound_results = fake_vector_search(COMPOUND_QUERY, top_k=3)
print('Top-3 retrieved documents:')
for rank, (doc_id, text) in enumerate(compound_results, 1):
    print(f'  {rank}. [{doc_id}] {text}')

print()
print('--- Problem ---')
print('The compound query retrieves a mish-mash.')
print('It may miss one competitor entirely, or retrieve shipping instead of refunds.')
print('The LLM gets confused context and produces a confused answer.')

print()
print('=== Focused queries (what agentic RAG does instead) ===')
for q in FOCUSED_QUERIES:
    results = fake_vector_search(q, top_k=1)
    doc_id, text = results[0]
    print(f'\n  Q: "{q}"')
    print(f'  Retrieved: [{doc_id}] {text}')

print()
print('Three focused queries, three precise hits.')
print('Standard RAG: 1 blurry query. Agentic RAG: N focused queries.')


---
## 2. Embedding Collapse on Multi-Intent Queries

Why does the compound query retrieve noise?

> **Embeddings represent the average meaning of the whole sentence.**
>
> Packing four intents into one question produces a vector that is near none of them.
> It is like searching for a book by averaging the titles of four different books.

Let's measure this. We compare cosine similarity between:
1. A **focused** query and its target document → should be high
2. The **compound** query and each target document → should be lower for each one

If the compound query's max similarity is lower than each focused query's similarity,
the embedding model has confirmed the problem.


In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print('Embedding model loaded.')

# Documents we care about
TARGET_DOCS = {
    'our_refund':   KNOWLEDGE_BASE['our_refund'],
    'compA_refund': KNOWLEDGE_BASE['compA_refund'],
    'compB_refund': KNOWLEDGE_BASE['compB_refund'],
}

# Queries
queries = {
    'Compound query': COMPOUND_QUERY,
    'Focused: ours':  FOCUSED_QUERIES[0],
    'Focused: comp A': FOCUSED_QUERIES[1],
    'Focused: comp B': FOCUSED_QUERIES[2],
}

doc_texts  = list(TARGET_DOCS.values())
doc_labels = list(TARGET_DOCS.keys())
doc_embeds = model.encode(doc_texts, convert_to_tensor=True, show_progress_bar=False)

print()
print(f'  {"Query":<28} ' + '  '.join(f'{d:<18}' for d in doc_labels))
print('-' * 90)

sim_matrix = {}
for q_label, q_text in queries.items():
    q_emb = model.encode(q_text, convert_to_tensor=True, show_progress_bar=False)
    sims  = util.cos_sim(q_emb, doc_embeds)[0].cpu().numpy()
    sim_matrix[q_label] = sims
    print(f'  {q_label:<28} ' + '  '.join(f'{s:<18.3f}' for s in sims))

print()
compound_max = max(sim_matrix['Compound query'])
focused_max  = max(max(sim_matrix[f]) for f in list(queries.keys())[1:])
print(f'Compound query  — best doc similarity: {compound_max:.3f}')
print(f'Focused queries — best doc similarity: {focused_max:.3f}')
print()
if compound_max < focused_max:
    print('Confirmed: compound query is BLURRIER than focused queries.')
    print('Each focused query finds its target more confidently.')
else:
    print('Result varies by embedding model — try a more complex compound question.')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

q_labels  = list(queries.keys())
doc_short = ['Our refund', 'Comp A refund', 'Comp B refund']
matrix    = np.array([sim_matrix[q] for q in q_labels])

# Heatmap
im = ax1.imshow(matrix, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax1.set_xticks(range(len(doc_short)))
ax1.set_xticklabels(doc_short, fontsize=10)
ax1.set_yticks(range(len(q_labels)))
ax1.set_yticklabels(q_labels, fontsize=9)
ax1.set_title('Cosine Similarity: Query vs. Document', fontweight='bold')
for i in range(len(q_labels)):
    for j in range(len(doc_short)):
        ax1.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center',
                 color='white' if matrix[i,j] > 0.55 else 'black', fontsize=10)
plt.colorbar(im, ax=ax1)

# Bar chart: max similarity per query
max_sims = [m.max() for m in [sim_matrix[q] for q in q_labels]]
bar_colors = ['#E53935'] + ['#1565C0'] * 3
bars = ax2.bar(range(len(q_labels)), max_sims, color=bar_colors, alpha=0.85, width=0.6)
ax2.set_xticks(range(len(q_labels)))
ax2.set_xticklabels([q.replace(': ', ':\n') for q in q_labels], fontsize=9)
ax2.set_ylim(0, 1.1)
ax2.set_ylabel('Max cosine similarity to best-matching doc')
ax2.set_title('How Focused vs. Compound Queries Compare', fontweight='bold')
for bar, val in zip(bars, max_sims):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02,
             f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
legend_handles = [
    mpatches.Patch(color='#E53935', label='Compound query (blurry)'),
    mpatches.Patch(color='#1565C0', label='Focused query (sharp)'),
]
ax2.legend(handles=legend_handles, loc='lower right')
plt.suptitle('Compound Queries Produce Lower Retrieval Confidence',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print('Key insight: focused queries give the embedding model a fighting chance.')
print('The agentic planner exists precisely to break compound queries into focused ones.')


---
## 3. Component 1 — The Planner

**Job:** Break the user's compound question into 2–4 focused, retrievable sub-questions.

> The planner makes or breaks everything downstream. If it misses a sub-question,
> no retrieval strategy can recover it. The final answer will be silently incomplete.

The planner is just an LLM call with a specific prompt.
It isn't a separate model — it's the same LLM wearing a different hat.

**Prompt pattern:**
```
Given the user's question, list the sub-questions you'd need to answer.
Output as a JSON array. Think carefully about every entity mentioned.
```

**Bad planning is the #1 silent failure mode in agentic RAG.**
Always include instructions like:
- *"Think carefully about every entity mentioned"*
- *"List all dimensions of comparison the user implies"*
- *"Output N sub-questions before stopping"*


In [ ]:
PLANNER_PROMPT = """\
Break the user's question into 2-4 focused sub-questions.
Each sub-question should be answerable by a single document lookup.
Think carefully about every entity mentioned and every comparison dimension implied.

Return ONLY a JSON object with key "sub_questions" whose value is an array of strings.

User question: {question}"""

# ── Mock planner (no API key needed) ─────────────────────────────────────────
MOCK_PLANS = {
    'default': [
        'What is our refund policy?',
        "What is Competitor A's refund policy?",
        "What is Competitor B's refund policy?",
        'On which dimensions should refund policies be compared (window, fees, conditions)?',
    ],
    'simple': [
        'What is our return window in days?',
    ],
    'pricing': [
        'What are our pricing tiers?',
        "What are Competitor A's pricing tiers?",
        'How do the prices compare?',
    ],
}


def planner(user_question: str) -> list:
    """Break user question into focused sub-questions."""
    if USE_REAL_LLM:
        client_llm = anthropic.Anthropic()
        prompt     = PLANNER_PROMPT.format(question=user_question)
        resp = client_llm.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=400,
            messages=[{'role': 'user', 'content': prompt}],
        )
        text = resp.content[0].text.strip()
        # Extract JSON from response
        start = text.find('{')
        end   = text.rfind('}') + 1
        data  = json.loads(text[start:end])
        return data.get('sub_questions', [])

    # Mock: keyword-based selection
    q_lower = user_question.lower()
    if 'refund' in q_lower and ('competitor' in q_lower or 'compare' in q_lower):
        return MOCK_PLANS['default']
    if 'pricing' in q_lower or 'price' in q_lower:
        return MOCK_PLANS['pricing']
    return MOCK_PLANS['simple']


print('=== Planner demo ===\n')
test_questions = [
    (
        'Compare our refund policy with Competitor A and Competitor B. Where are we stricter?',
        'compound-comparison'
    ),
    ('What is our return window?', 'simple'),
]
for q, note in test_questions:
    plan = planner(q)
    print(f'Question ({note}): "{q}"')
    print(f'Sub-questions ({len(plan)}):')
    for i, sq in enumerate(plan, 1):
        print(f'  {i}. {sq}')
    print()

print('Observe: the compound question produces 4 sub-questions;')
print('the simple question produces 1. The agent does proportional work.')


---
## 4. Component 2 — The Retriever

**Job:** For each sub-question, fetch relevant chunks. The same retriever you already know —
but called **N times** with **N focused queries** instead of once with a blurry compound query.

The retriever has two parts:

1. **Query formulation:** rewrite the sub-question into an optimal search query
   (shorter, keyword-dense, stripped of conversational filler)
2. **Vector search:** run the query against the index

```
Sub-question: "What is Competitor A's refund window in days?"
      ↓  query formulation
Query: "Competitor A refund window days"
      ↓  vector search
Chunks: [compA_refund: "30-day refund window, 10% restocking fee..."]
```

Query formulation is a separate LLM call. It exists because sub-questions often contain
phrasing that is good for humans but bad for embedding search ("What is...", "Tell me about...").


In [ ]:
QUERY_FORMULATION_PROMPT = """\
Rewrite this question as a short keyword search query (5 words max).
Remove conversational filler. Keep only the key entities and attributes.
Return ONLY the query, nothing else.

Question: {question}"""

# ── Mock query formulation ────────────────────────────────────────────────────
MOCK_QUERIES = {
    'What is our refund policy?':                   'our refund policy',
    "What is Competitor A's refund policy?":         'Competitor A refund policy',
    "What is Competitor B's refund policy?":         'Competitor B refund policy',
    'On which dimensions should refund policies be compared (window, fees, conditions)?':
                                                     'refund window fees conditions',
    'What is our return window in days?':            'our refund return days',
    'What are our pricing tiers?':                   'pricing tiers plans',
    "What are Competitor A's pricing tiers?":        'Competitor A pricing tiers',
    'How do the prices compare?':                   'pricing comparison tiers',
}


def formulate_query(sub_question: str) -> str:
    """Rewrite a sub-question into a short keyword search query."""
    if USE_REAL_LLM:
        client_llm = anthropic.Anthropic()
        prompt     = QUERY_FORMULATION_PROMPT.format(question=sub_question)
        resp = client_llm.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=30,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return resp.content[0].text.strip()

    return MOCK_QUERIES.get(sub_question, sub_question[:40])


def retrieve(sub_question: str, top_k: int = 2) -> list:
    """Formulate a query from the sub-question, then search."""
    search_query = formulate_query(sub_question)
    results      = fake_vector_search(search_query, top_k=top_k)
    return search_query, results


print('=== Retriever demo ===\n')
plan = planner(COMPOUND_QUERY)
print(f'Plan: {len(plan)} sub-questions\n')
for sq in plan:
    search_q, results = retrieve(sq)
    print(f'Sub-question: "{sq}"')
    print(f'  Formulated query: "{search_q}"')
    for doc_id, text in results:
        print(f'  [{doc_id}] {text[:80]}...')
    print()

print('Each focused query retrieves its target document precisely.')
print('Compare this to the compound query which retrieved a mish-mash in Section 1.')


---
## 5. Component 3 — The Critic

**Job:** After each retrieval, ask: *"Is this enough? Is it trustworthy? Did I miss anything?"*

This is **the component that doesn't exist in standard RAG.**
The critic is what lets the system decide whether to keep going or stop.

```
Critic: "I have our policy and Competitor A's, but Competitor B's chunk
         only mentions 'returns,' not 'refunds.' Let me try again."
```

**Implementation:** another LLM prompt that rates sufficiency 1–5 and explains what's missing.
If the score is low, the loop continues with a refined query.

> **Always pair the critic with a hard ceiling.**
> A confused agent can get stuck in "not quite enough, try again" loops.
> Add `max_retries=5` or a cost cap. Without it, a single query can rack up surprising bills.


In [ ]:
CRITIC_PROMPT = """\
You are evaluating retrieved evidence for a RAG system.

Sub-question: {sub_question}
Retrieved chunks:
{chunks}

Is this sufficient to answer the sub-question?
Rate sufficiency 1-5 (5 = fully sufficient, 1 = completely missing).

Return ONLY JSON: {{"sufficient": true/false, "score": 1-5, "reason": "..."}}"""

# ── Mock critic ───────────────────────────────────────────────────────────────
# Reflects the blog's logic: passes on good docs, fails if text is off-topic.
CRITIC_PASS_KEYWORDS = {
    'What is our refund policy?':                ['our_refund'],
    "What is Competitor A's refund policy?":      ['compA_refund'],
    "What is Competitor B's refund policy?":      ['compB_refund'],
    'On which dimensions should refund policies be compared (window, fees, conditions)?':
                                                  ['our_refund', 'compA_refund', 'compB_refund'],
    'What is our return window in days?':        ['our_refund'],
    'What are our pricing tiers?':               ['pricing'],
    "What are Competitor A's pricing tiers?":    ['compA_refund'],  # simulated miss on first try
    'How do the prices compare?':               ['pricing'],
}


def critic(sub_question: str, results: list) -> dict:
    """Judge whether retrieved chunks sufficiently answer the sub-question."""
    if USE_REAL_LLM:
        client_llm  = anthropic.Anthropic()
        chunk_text  = chr(10).join(f'- [{did}] {text}' for did, text in results)
        prompt      = CRITIC_PROMPT.format(
            sub_question=sub_question, chunks=chunk_text)
        resp = client_llm.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=120,
            messages=[{'role': 'user', 'content': prompt}],
        )
        text  = resp.content[0].text.strip()
        start = text.find('{')
        end   = text.rfind('}') + 1
        return json.loads(text[start:end])

    # Mock: check if any retrieved doc_id is in the expected list
    expected = CRITIC_PASS_KEYWORDS.get(sub_question, [])
    got_ids  = [did for did, _ in results]
    hits     = [did for did in got_ids if did in expected]
    if hits:
        return {'sufficient': True,  'score': 5,
                'reason': f'Found relevant document(s): {hits}'}
    return {'sufficient': False, 'score': 2,
            'reason': 'Retrieved documents do not match the sub-question. Try a different query.'}


print('=== Critic demo ===\n')
demo_cases = [
    ('What is our refund policy?',
     [('our_refund', KNOWLEDGE_BASE['our_refund'])],
     'Correct doc retrieved'),
    ("What is Competitor A's refund policy?",
     [('shipping', KNOWLEDGE_BASE['shipping'])],
     'Wrong doc retrieved (simulated retrieval failure)'),
]
for sq, results, note in demo_cases:
    verdict = critic(sq, results)
    print(f'Scenario: {note}')
    print(f'  Sub-question: "{sq}"')
    print(f'  Retrieved   : {[did for did, _ in results]}')
    print(f'  Verdict     : sufficient={verdict["sufficient"]}  '
          f'score={verdict["score"]}  reason="{verdict["reason"]}"')
    print()

print('When sufficient=False, the agent retries with a refined query.')
print('This feedback loop is the core of what makes agentic RAG adaptive.')


---
## 6. Component 4 — The Synthesiser

**Job:** Take all the gathered evidence and write the final answer.

Same generator as standard RAG — but it now receives **labelled, organised context**,
not a random pile of chunks.

```
### Evidence for: What is our refund policy?
- Our refund policy: refunds within 14 days. No restocking fee.

### Evidence for: What is Competitor A's refund policy?
- Competitor A: 30-day refund window. 10% restocking fee on opened items.

### Evidence for: What is Competitor B's refund policy?
- Competitor B: 7-day refund window. Refunds only for defective items.
```

**Why labelling the context matters:**
Dumping all chunks into one blob forces the LLM to figure out which chunk answers which
sub-question — and it often mixes them up. Headers like `### Our refund policy:` are the
difference between a clean comparison table and a confused paragraph.


In [ ]:
SYNTHESISER_PROMPT = """\
Answer the user's question using ONLY the evidence below.
Be specific and cite which sub-topic each fact comes from.
Structure the answer as a clear comparison.

User question: {question}

{evidence}"""

# ── Mock synthesiser ──────────────────────────────────────────────────────────
MOCK_SYNTHESIS = (
    """Based on the evidence gathered:\n\n"""
    """**Refund windows:**\n"""
    """- Our policy: 14 days\n"""
    """- Competitor A: 30 days (more lenient than us)\n"""
    """- Competitor B: 7 days, defects only (stricter than us)\n\n"""
    """**Restocking fees:**\n"""
    """- Our policy: none\n"""
    """- Competitor A: 10% on opened items (stricter than us)\n"""
    """- Competitor B: N/A\n\n"""
    """**Where we are stricter:** We are stricter than Competitor A on the refund window """
    """(14 days vs. 30 days). We are more lenient than Competitor B on scope """
    """(all products vs. defects only). We have no restocking fee, which is """
    """more lenient than Competitor A."""
)


def synthesiser(user_question: str, evidence: dict) -> str:
    """Combine evidence from all sub-questions into the final answer."""
    # Build labelled context (the key insight from the blog)
    formatted = chr(10).join(
        f'### Evidence for: {sq}\n' + chr(10).join(f'- [{did}] {text}' for did, text in chunks)
        for sq, chunks in evidence.items()
    )
    if USE_REAL_LLM:
        client_llm = anthropic.Anthropic()
        prompt     = SYNTHESISER_PROMPT.format(
            question=user_question, evidence=formatted)
        resp = client_llm.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=600,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return resp.content[0].text.strip()

    return MOCK_SYNTHESIS


# Demo: contrast labelled vs. unlabelled context
demo_evidence = {
    'What is our refund policy?':
        [('our_refund', KNOWLEDGE_BASE['our_refund'])],
    "What is Competitor A's refund policy?":
        [('compA_refund', KNOWLEDGE_BASE['compA_refund'])],
    "What is Competitor B's refund policy?":
        [('compB_refund', KNOWLEDGE_BASE['compB_refund'])],
}

print('=== Synthesiser context format ===\n')
print('What the synthesiser receives (labelled, organised):')
print('---')
for sq, chunks in demo_evidence.items():
    print(f'### Evidence for: {sq}')
    for did, text in chunks:
        print(f'- [{did}] {text}')
    print()
print('---')
print()
print('What standard RAG sends (unlabelled blob):')
print('---')
all_text = ' '.join(text for chunks in demo_evidence.values() for _, text in chunks)
print(all_text)
print('---')
print()
print('The labels help the LLM attribute each fact to the right sub-topic.')
print('Without them, it will mix up our policy with Competitor A and produce wrong comparisons.')


---
## 7. The Agent Loop

All four components wired together. This is the agentic RAG loop:

```
1. PLAN   — planner breaks the question into sub-questions
      ↓
2. RETRIEVE — retriever fetches chunks for each sub-question
      ↓
3. CRITIQUE — critic judges sufficiency
      ↓ (if insufficient, retry with refined query — up to max_retries)
4. SYNTHESISE — synthesiser writes the final answer from labelled evidence
```

**The `max_retries` ceiling is non-negotiable.**
Without it, a confused critic can trap the agent in an infinite retry loop.
In production, add a cost cap alongside the retry ceiling.


In [ ]:
TRACE = []  # Global trace log; populated by agentic_rag()


def agentic_rag(user_question: str, max_retries: int = 2) -> str:
    global TRACE
    TRACE = []
    llm_calls = 0

    TRACE.append({'step': 'question', 'content': user_question})
    print(f'USER: {user_question}\n')

    # ── Step 1: Plan ─────────────────────────────────────────────────────────
    sub_questions = planner(user_question)
    llm_calls    += 1
    TRACE.append({'step': 'plan', 'content': sub_questions, 'llm_calls': llm_calls})
    print(f'PLAN ({len(sub_questions)} sub-questions):')
    for i, sq in enumerate(sub_questions, 1):
        print(f'  {i}. {sq}')
    print()

    # ── Steps 2-3: Retrieve + Critic loop ────────────────────────────────────
    evidence = {}
    for sq in sub_questions:
        attempt   = 0
        sq_query, chunks = retrieve(sq)
        llm_calls += 1  # query formulation

        verdict   = critic(sq, chunks)
        llm_calls += 1  # critic call

        status = 'OK' if verdict['sufficient'] else 'RETRY'
        TRACE.append({
            'step': 'retrieve',
            'sub_question': sq,
            'query': sq_query,
            'docs': [did for did, _ in chunks],
            'verdict': verdict,
            'attempt': attempt,
            'llm_calls': llm_calls,
        })
        print(f'RETRIEVE "{sq[:50]}..." '  if len(sq) > 50 else f'RETRIEVE "{sq}" ')
        print(f'  query="{sq_query}"  docs={[d for d,_ in chunks]}  '
              f'critic={status}  score={verdict["score"]}')

        # Retry loop
        while not verdict['sufficient'] and attempt < max_retries:
            attempt  += 1
            hint      = sq + ' ' + verdict['reason']
            re_q, chunks = retrieve(hint)
            llm_calls += 1

            verdict   = critic(sq, chunks)
            llm_calls += 1

            status    = 'OK' if verdict['sufficient'] else 'RETRY'
            TRACE.append({
                'step': 'retry',
                'sub_question': sq,
                'attempt': attempt,
                'query': re_q,
                'docs': [did for did, _ in chunks],
                'verdict': verdict,
                'llm_calls': llm_calls,
            })
            print(f'  RETRY #{attempt}  query="{re_q}"  '
                  f'docs={[d for d,_ in chunks]}  critic={status}')

        evidence[sq] = chunks

    # ── Step 4: Synthesise ────────────────────────────────────────────────────
    print()
    print('SYNTHESISING...')
    answer    = synthesiser(user_question, evidence)
    llm_calls += 1
    TRACE.append({'step': 'synthesise', 'llm_calls': llm_calls})
    TRACE.append({'step': 'answer', 'content': answer, 'total_llm_calls': llm_calls})

    print(f'\nANSWER (after {llm_calls} LLM calls):')
    print('=' * 60)
    print(answer)
    print('=' * 60)
    return answer


In [ ]:
# Run the full agent loop
answer = agentic_rag(
    'Compare our refund policy with Competitor A and Competitor B. '
    'Where are we stricter?',
    max_retries=2
)


---
## 8. Execution Trace Visualisation

The trace lets you inspect exactly what the agent did, in what order, and how many LLM calls
each step consumed. This is your debugging dashboard.


In [ ]:
# ── Execution trace: horizontal waterfall ────────────────────────────────────
if not TRACE:
    print('Run cell-19 first to populate the trace.')
else:
    retrieve_steps = [t for t in TRACE if t['step'] in ('retrieve', 'retry')]
    total_calls    = TRACE[-1].get('total_llm_calls', 0)

    step_labels = []
    step_types  = []
    for t in TRACE:
        if t['step'] == 'plan':
            step_labels.append('Plan')
            step_types.append('plan')
        elif t['step'] == 'retrieve':
            sq_short = t['sub_question'][:30] + '...' if len(t['sub_question']) > 30 else t['sub_question']
            step_labels.append(f'Retrieve\n"{sq_short}"')
            step_types.append('retrieve')
        elif t['step'] == 'retry':
            step_labels.append(f'Retry #{t["attempt"]}')
            step_types.append('retry')
        elif t['step'] == 'synthesise':
            step_labels.append('Synthesise')
            step_types.append('synthesise')

    color_map = {'plan': '#1565C0', 'retrieve': '#2E7D32',
                 'retry': '#E65100', 'synthesise': '#6A1B9A'}
    colors = [color_map.get(t, 'gray') for t in step_types]

    fig, ax = plt.subplots(figsize=(max(14, len(step_labels)*2), 4))
    x_pos = range(len(step_labels))
    bars  = ax.bar(x_pos, [1]*len(step_labels), color=colors, alpha=0.85, width=0.7)
    ax.set_xticks(list(x_pos))
    ax.set_xticklabels(step_labels, fontsize=8, rotation=15, ha='right')
    ax.set_yticks([])
    ax.set_title(
        f'Agentic RAG Execution Trace — {total_calls} LLM calls total\n'
        '(Standard RAG: 1 LLM call)',
        fontweight='bold', fontsize=11)

    legend_handles = [
        mpatches.Patch(color=v, label=k.capitalize()) for k, v in color_map.items()
    ]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    plt.tight_layout()
    plt.show()

    print(f'Total LLM calls: {total_calls}')
    print(f'Standard RAG equivalent: 1 LLM call')
    print(f'Overhead factor: {total_calls}x')
    print()
    print('LLM call breakdown:')
    print('  1 planner call')
    n_sub = len([t for t in TRACE if t['step'] == 'retrieve'])
    n_retry = len([t for t in TRACE if t['step'] == 'retry'])
    print(f'  {n_sub} query-formulation calls (one per sub-question)')
    print(f'  {n_sub} critic calls (one per retrieval)')
    if n_retry:
        print(f'  {n_retry} retry cycles (query-formulation + critic each)')
    print(f'  1 synthesiser call')
    print(f'  = {total_calls} total')


---
## 9. When to Use Agentic RAG

**Use agentic RAG when:** the question has structure that demands multiple lookups.

| Use case | Why agentic |
|---|---|
| Research synthesis | "What does the literature say about X, and where do experts disagree?" |
| Document comparison | "How does Contract A differ from Contract B?" |
| Multi-step analysis | "Find all customers who churned after pricing changes and summarise their feedback." |
| Exhaustive retrieval | "Tell me everything relevant about X from these 500 documents." |
| Ambiguous questions | "Why did our revenue dip in Q3?" (requires multiple data sources) |

**Skip agentic RAG when:**

| Use case | Why standard RAG is enough |
|---|---|
| Simple Q&A | "What's our return window?" — one lookup, one answer |
| Narrow chatbots | Pizza ordering bot doesn't need to plan |
| Latency-sensitive apps | Voice assistants, autocomplete — users won't wait 15 seconds |
| High-volume, low-margin | Free-tier search hit a million times a day |

> **The common thread:** if you can predict the retrieval pattern in advance, you don't need an agent.
> Just write the pipeline.


In [ ]:
def classify_query_complexity(question: str) -> dict:
    """
    Heuristic classifier: should this question use agentic or standard RAG?
    In production, replace with a lightweight LLM call or a fine-tuned classifier.
    """
    q = question.lower()

    # Strong agentic signals
    COMPARISON_WORDS = ['compare', 'versus', 'vs', 'differ', 'difference',
                        'stricter', 'better', 'worse', 'contrast']
    MULTI_ENTITY     = ['competitor', 'contract', 'vendor', 'all', 'each',
                        'every', 'across', 'multiple']
    WHY_HOW          = ['why', 'how did', 'how does', 'what caused', 'summarise',
                        'summarize', 'everything']

    comparison_hit = any(w in q for w in COMPARISON_WORDS)
    multi_hit      = any(w in q for w in MULTI_ENTITY)
    why_hit        = any(w in q for w in WHY_HOW)
    entity_count   = sum(1 for w in ['competitor a', 'competitor b', 'vendor',
                                      'contract', 'q1', 'q2', 'q3', 'q4'] if w in q)

    score = (comparison_hit * 3) + (multi_hit * 2) + (why_hit * 2) + (entity_count * 2)

    if score >= 5:
        path    = 'agentic'
        reason  = 'Multi-entity comparison or multi-step analysis detected'
    elif score >= 2:
        path    = 'agentic'
        reason  = 'Some complexity detected; agentic path adds safety'
    else:
        path    = 'standard'
        reason  = 'Simple lookup — standard RAG is sufficient'

    return {
        'question': question,
        'path':     path,
        'score':    score,
        'reason':   reason,
    }


TEST_QUESTIONS = [
    'What is our return window?',
    'What time does support close?',
    'Compare our refund policy with Competitor A and Competitor B.',
    'Why did our revenue dip in Q3?',
    'Summarise the key risks across all vendor contracts.',
    'What is the status of order #12345?',
    'How does our SLA differ from the enterprise tier SLA?',
]

print(f'{"Question":<60} {"Path":<10} {"Score"}')
print('-' * 80)
for q in TEST_QUESTIONS:
    result = classify_query_complexity(q)
    print(f'{q[:58]:<60} {result["path"]:<10} {result["score"]}')

print()
print('Note: this is a heuristic. In production, use a lightweight LLM call:')
print('  "Is this question answerable with a single document lookup? Answer yes/no."')
print('  3-5 tokens, < 50ms, routes correctly 95%+ of the time.')


---
## 10. Cost Comparison — Standard vs. Agentic RAG

Agentic RAG is expensive. Let's do the math.

For a single agentic query with 3 sub-questions and no retries:

| Step | LLM calls | Notes |
|---|---|---|
| Planner | 1 | Break question into sub-questions |
| Query formulation | 3 | One per sub-question |
| Critic | 3 | One after each retrieval |
| Synthesiser | 1 | Final answer |
| **Total** | **8** | **vs. 1 for standard RAG** |

Each subsequent retry adds 2 more calls (query formulation + critic).

> **Rule of thumb: Agentic RAG is 5–10× more expensive per query than standard RAG.**
>
> This isn't a reason to avoid it. It's a reason to be intentional.
> Save the agents for questions that actually need them.


In [ ]:
# ── LLM call count by scenario ───────────────────────────────────────────────
scenarios = [
    ('Standard RAG\n(any question)',     1,  '#4CAF50'),
    ('Agentic RAG\n2 sub-questions\n0 retries', 1+2+2+1, '#1565C0'),
    ('Agentic RAG\n3 sub-questions\n0 retries', 1+3+3+1, '#1976D2'),
    ('Agentic RAG\n3 sub-questions\n1 retry',   1+3+3+1+2, '#1E88E5'),
    ('Agentic RAG\n4 sub-questions\n2 retries', 1+4+4+1+4, '#42A5F5'),
]

labels = [s[0] for s in scenarios]
counts = [s[1] for s in scenarios]
colors = [s[2] for s in scenarios]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
bars = ax1.bar(range(len(scenarios)), counts, color=colors, alpha=0.85, width=0.6)
ax1.set_xticks(range(len(scenarios)))
ax1.set_xticklabels(labels, fontsize=8)
ax1.set_ylabel('LLM calls per user query')
ax1.set_title('LLM Calls: Standard vs. Agentic RAG', fontweight='bold')
for bar, val in zip(bars, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.1,
             str(val), ha='center', fontsize=12, fontweight='bold')

# Cost multiplier at $0.005 per call (rough estimate)
cost_per_call  = 0.005  # USD
costs          = [c * cost_per_call for c in counts]
ax2.bar(range(len(scenarios)), costs, color=colors, alpha=0.85, width=0.6)
ax2.set_xticks(range(len(scenarios)))
ax2.set_xticklabels(labels, fontsize=8)
ax2.set_ylabel('Estimated cost per query (USD, rough)')
ax2.set_title(f'Relative Cost (assuming ${cost_per_call}/LLM call)', fontweight='bold')
for i, (bar, val) in enumerate(zip(ax2.patches, costs)):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.0002,
             f'${val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Why Agentic RAG Demands Intentional Routing\n'
             '(only use it for questions that genuinely need it)',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

print('Cost note: actual cost depends on model choice, token counts, and caching.')
print('The call count ratio is reliable. The dollar amounts are illustrative.')
print()
print('Practical strategy: tier your routing system.')
print('  Easy questions → standard RAG (fast, cheap)')
print('  Hard questions → agentic RAG (slow, thorough, expensive)')


---
## 11. Routing Classifier — Pick the Right Path

Production systems don't choose one architecture. They route:

```
User query
    ↓
  [Router]
  /       \
standard   agentic
RAG        RAG
```

The router can be:
- A heuristic (keyword scoring) — < 1ms, shown in Section 9
- A small classification model — 5–50ms, more accurate
- An LLM call with a binary prompt — 200–500ms, most accurate, adds cost

Here we build the **full routing pipeline**: classify → dispatch → answer.


In [ ]:
def standard_rag(user_question: str) -> dict:
    """Standard RAG: one query, one retrieval, one LLM call."""
    _, chunks = retrieve(user_question, top_k=3)[1], retrieve(user_question, top_k=3)
    sq_query, results = retrieve(user_question, top_k=3)
    context = chr(10).join(f'- [{did}] {text}' for did, text in results)

    if USE_REAL_LLM:
        client_llm = anthropic.Anthropic()
        resp = client_llm.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=400,
            messages=[{
                'role': 'user',
                'content': f'Answer using ONLY this context:\n{context}\n\nQuestion: {user_question}'
            }],
        )
        answer = resp.content[0].text.strip()
    else:
        # Mock: return the raw context as the answer
        answer = f'Based on retrieved documents:\n{context}'

    return {'answer': answer, 'llm_calls': 1, 'path': 'standard'}


def routed_rag(user_question: str) -> dict:
    """Full routing pipeline: classify → dispatch → answer."""
    classification = classify_query_complexity(user_question)
    path           = classification['path']
    reason         = classification['reason']

    print(f'[Router] path={path}  reason="{reason}"')

    if path == 'agentic':
        answer = agentic_rag(user_question, max_retries=2)
        return {'answer': answer, 'path': 'agentic'}
    else:
        result = standard_rag(user_question)
        print(f'Answer (standard RAG, 1 LLM call):\n{result["answer"]}')
        return result


print('=== Routing demo ===\n')
routing_demos = [
    'What is our return window?',
    'Compare our refund policy with Competitor A and Competitor B. Where are we stricter?',
]
for q in routing_demos:
    print(f'Question: "{q}"')
    result = routed_rag(q)
    print(f'Path taken: {result["path"]}')
    print()
    print('---' * 20)
    print()


---
## 12. Claude API Version

Set `USE_REAL_LLM = True` in cell-02 and add your API key, then re-run cells 10–19.
All four components (planner, retriever, critic, synthesiser) will use live Claude calls.

**Model selection guide for agentic RAG components:**

| Component | Recommended model | Reason |
|---|---|---|
| Planner | `claude-sonnet-4-6` | Needs nuanced understanding of compound questions |
| Query formulation | `claude-haiku-4-5-20251001` | Simple rewrite task; fast + cheap |
| Critic | `claude-haiku-4-5-20251001` | Binary yes/no + reason; fast enough |
| Synthesiser | `claude-sonnet-4-6` | Complex structured output; quality matters most |

Mixing models by task is a cost optimisation strategy:
you pay for intelligence only where it matters.


In [ ]:
# ── Live Claude API version ──────────────────────────────────────────────────
# Enable: set USE_REAL_LLM = True in cell-02 + set ANTHROPIC_API_KEY

if USE_REAL_LLM and ANTHROPIC_AVAILABLE:
    print('Running full agentic RAG pipeline with live Claude API...\n')
    final_answer = agentic_rag(
        'Compare our refund policy with Competitor A and Competitor B. '
        'Where are we stricter?',
        max_retries=2,
    )
else:
    print('ANTHROPIC_API_KEY not set or USE_REAL_LLM=False.')
    print()
    print('What changes when you enable live mode:')
    print()
    print('1. PLANNER — receives PLANNER_PROMPT, returns a JSON sub-question list.')
    print('   Haiku will catch entities you might miss in a heuristic planner.')
    print()
    print('2. QUERY FORMULATION — receives QUERY_FORMULATION_PROMPT.')
    print('   "Tell me about Competitor A refund" → "Competitor A refund window"')
    print('   The model strips filler words and keeps only embedding-friendly keywords.')
    print()
    print('3. CRITIC — receives CRITIC_PROMPT with retrieved chunks.')
    print('   Returns {"sufficient": bool, "score": 1-5, "reason": str}.')
    print('   The reason field is what the retry loop uses as a hint.')
    print()
    print('4. SYNTHESISER — receives SYNTHESISER_PROMPT with labelled evidence.')
    print('   Produces a structured comparison with citations to sub-topics.')
    print()
    print('Model mix recommended:')
    print('  Planner    → claude-sonnet-4-6      (complex decomposition)')
    print('  Query form → claude-haiku-4-5-20251001 (simple, fast, cheap)')
    print('  Critic     → claude-haiku-4-5-20251001 (yes/no + reason)')
    print('  Synthesise → claude-sonnet-4-6      (quality output)')


---
## Key Takeaways

1. **Standard RAG fails on compound questions** because embeddings average meaning across intents.
   "Compare our policy with two competitors" produces a blurry vector that is close to nothing in
   your index. The fix is to decompose the question before embedding, not after.

2. **Retrieval as a tool means the LLM decides when and what to retrieve.**
   In standard RAG, the developer hardcodes the retrieval. In agentic RAG, the LLM calls
   retrieval as a function, reads the result, and decides whether to call it again.

3. **Four roles, usually one model.** Planner, retriever, critic, synthesiser are conceptual
   roles, not separate models. They are the same LLM called with different prompts.
   Mix models only when cost optimisation matters.

4. **The critic is what makes the loop adaptive.** Without it, the agent has no feedback.
   It either stops too early (and hallucinates from incomplete context) or loops forever.
   Always pair the critic with a hard ceiling (`max_retries`, cost cap).

5. **Label your context before synthesis.** Passing a structured block
   (`### Evidence for: <sub-question>`) to the synthesiser produces coherent comparisons.
   Passing a blob produces a confused summary that mixes up entities.

6. **Agentic RAG is 5–10× more expensive per query.** Not a reason to avoid it — a reason
   to route intentionally. Build a classifier that sends simple questions to standard RAG
   and compound questions to the agent. Most production traffic is simple.

7. **Bad planning is the #1 silent failure mode.** If the planner misses Competitor B,
   the answer will be silently incomplete. No warning, no error — just a wrong answer delivered
   confidently. Planner prompts deserve the most careful engineering.

---

*Up next — Lesson 10.2: Multi-query retrieval and query expansion inside the agent loop.*
